# Notebook 01: Translation PoC (IndicTrans2)

This notebook verifies the translation stage of the IndicVoice pipeline using `ai4bharat/indictrans2-en-indic-dist-200M`.
Target languages tested: **Odia (ory_Orya)**, **Hindi (hin_Deva)**, and **Bengali (ben_Beng)**.

In [ ]:
# Install dependencies (run once in Colab/Kaggle)
!pip install -q transformers sentencepiece protobuf accelerate
!pip install -q git+https://github.com/AI4Bharat/IndicTransToolkit.git

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if "cuda" in DEVICE else torch.float32
MODEL_ID = "ai4bharat/indictrans2-en-indic-dist-200M"

print(f"Using device: {DEVICE} ({DTYPE})")

In [ ]:
# Load model and tokenizer
print("Loading tokenizer and IndicTrans2 weights...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DTYPE
).to(DEVICE)
model.eval()
ip = IndicProcessor(inference=True)
print("Model loaded successfully!")

In [ ]:
def translate_batch(sentences, target_lang_code="ory_Orya"):
    preprocessed = ip.preprocess_batch(sentences, src_lang="eng_Latn", tgt_lang=target_lang_code)
    inputs = tokenizer(preprocessed, padding="longest", truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    
    with torch.inference_mode():
        outputs = model.generate(**inputs, num_beams=5, max_length=256)
        
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return ip.postprocess_batch(decoded, lang=target_lang_code)

# Test sentences
test_sentences = [
    "Welcome to Odisha.",
    "Bhubaneswar is known as the temple city of India.",
    "The sun rises in the east and sets in the west.",
    "What time does the train arrive in Cuttack?",
    "Have a wonderful day ahead!"
]

for lang_name, code in [("Odia", "ory_Orya"), ("Hindi", "hin_Deva"), ("Bengali", "ben_Beng")]:
    print(f"\n--- {lang_name} ({code}) ---")
    results = translate_batch(test_sentences, target_lang_code=code)
    for en, tr in zip(test_sentences, results):
        print(f"EN: {en}")
        print(f"{lang_name[:2].upper()}: {tr}\n")